# 🐍 Notebook 07: Python Functions, Decorators & Lambda Expressions
**Trainer: Sundeep Hiremat · TASK Nalgonda Center**  
**Course:** Cloud Software Developer (AWS DVA-C02 Aligned)

In this notebook, we master three critical building blocks of cloud programming:
1. **Functions, `*args`, and `**kwargs`:** How AWS SDKs (`boto3`) accept dynamic configurations.
2. **Lambda Functions (Anonymous Functions):** Quick, inline logic for sorting, filtering, and mapping.
3. **Decorators (`@decorator`):** Clean, reusable wrappers for logging, retrying failed AWS calls, and timing latency.


## 1. Functions & Dynamic Arguments (`*args`, `**kwargs`)
In AWS development, functions often need to accept dynamic options (like tags, metadata, or pagination parameters).


In [ ]:
# 1. Standard Function with Default & Type Hints
def calculate_instance_cost(hours: int, rate_per_hour: float = 0.0208) -> float:
    """Calculates total run cost for an EC2 t3.micro instance."""
    total = hours * rate_per_hour
    return round(total, 4)

print("Cost for 24 hours:", calculate_instance_cost(24))
print("Cost for 720 hours (1 month):", calculate_instance_cost(720))


### Variable Arguments: `*args` (Tuple) and `**kwargs` (Dictionary)
* `*args`: Collects extra positional arguments into a **tuple**.
* `**kwargs`: Collects extra keyword arguments into a **dictionary**.


In [ ]:
def deploy_cloud_service(service_name, region="ap-south-1", *tags, **metadata):
    print(f"🚀 Deploying: {service_name} in {region}")
    print(f"📌 Positional Tags (tuple): {tags}")
    print(f"⚙️ Metadata Options (dict): {metadata}")

# Calling with variable arguments
deploy_cloud_service(
    "Amazon EC2",
    "us-east-1",
    "Production", "WebTier", "Nalgonda",
    Owner="Sundeep", InstanceType="t3.medium", AutoTerminate=True
)


---
## 2. Lambda Functions (Anonymous Functions)
A **lambda function** is a small, one-line function without a name.  
$$\text{Syntax: } \text{lambda } arguments : expression$$

> 💡 **Notice:** This is a Python language feature! Don't confuse it with **AWS Lambda** (the cloud compute service).


In [ ]:
# Regular function vs Lambda
def add_tax(price):
    return price * 1.18

# Same logic in a lambda
add_tax_lambda = lambda price: price * 1.18

print("Regular function:", add_tax(100))
print("Lambda function:", add_tax_lambda(100))


### Real Cloud Use Case: Sorting Cloud Fleets by Cost
In cloud engineering, we frequently fetch lists of servers from AWS and sort them by price, CPU, or launch time:


In [ ]:
# List of running EC2 instances
ec2_inventory = [
    {"instance_id": "i-001", "type": "t3.large", "hourly_usd": 0.0832, "state": "running"},
    {"instance_id": "i-002", "type": "t2.nano",  "hourly_usd": 0.0058, "state": "stopped"},
    {"instance_id": "i-003", "type": "t3.micro", "hourly_usd": 0.0208, "state": "running"},
    {"instance_id": "i-004", "type": "c5.xlarge", "hourly_usd": 0.1700, "state": "running"}
]

# 1. Sort by hourly cost ascending using a lambda
sorted_by_cost = sorted(ec2_inventory, key=lambda inst: inst["hourly_usd"])

print("EC2 Fleet sorted by cost (cheapest first):")
for inst in sorted_by_cost:
    print(f"  • {inst['instance_id']} ({inst['type']}): ${inst['hourly_usd']}/hr")


### Filtering with Lambdas (`filter` and List Comprehensions)


In [ ]:
# Filter only RUNNING instances using lambda
running_instances = list(filter(lambda inst: inst["state"] == "running", ec2_inventory))
print("Running instances:", [i["instance_id"] for i in running_instances])

# Equivalent list comprehension (very idiomatic in Python)
running_comp = [i["instance_id"] for i in ec2_inventory if i["state"] == "running"]
print("Using list comprehension:", running_comp)


---
## 3. Python Decorators (`@decorator`)
A **decorator** is a function that takes another function, adds behavior to it (like logging, timing, or authentication), and returns it.

### Step 1: Writing a Latency/Timer Decorator


In [ ]:
import time
import functools

def timer_decorator(func):
    """Measures and prints execution time of any function."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        duration_ms = (time.time() - start) * 1000
        print(f"⏱️ [CloudWatch Metric] '{func.__name__}' took {duration_ms:.2f} ms")
        return result
    return wrapper

# Using the decorator with '@'
@timer_decorator
def sync_s3_bucket(bucket_name, total_files=10):
    print(f"Starting synchronization of s3://{bucket_name}...")
    time.sleep(0.3) # simulate network transfer
    return f"Synced {total_files} files successfully."

# Run it
response = sync_s3_bucket("task-nalgonda-assets", total_files=25)
print("Response:", response)


### Step 2: Cloud Resilience Decorator: `@retry_aws_call`
In AWS, API calls can be throttled. This decorator automatically retries a failed function:


In [ ]:
def retry_aws_call(max_attempts=3, delay_sec=1):
    """Decorator that retries a function if it raises an exception."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            attempts = 0
            while attempts < max_attempts:
                try:
                    return func(*args, **kwargs)
                except Exception as err:
                    attempts += 1
                    print(f"⚠️ Attempt {attempts} failed: {err}. Retrying in {delay_sec}s...")
                    time.sleep(delay_sec)
            raise RuntimeError(f"❌ '{func.__name__}' failed after {max_attempts} attempts.")
        return wrapper
    return decorator

# Simulating an API call that fails twice then succeeds
call_count = 0

@retry_aws_call(max_attempts=4, delay_sec=1)
def fetch_dynamodb_record(user_id):
    global call_count
    call_count += 1
    if call_count < 3:
        raise ConnectionError("AWS DynamoDB RequestLimitExceeded (429)")
    return {"user_id": user_id, "name": "B. Sai Kiran", "status": "Active"}

# Test the resilience
record = fetch_dynamodb_record("USR-101")
print("🎉 Final Result:", record)
